# Descargar Raster Global Wind Atlas (GWA) — Costa Rica

Este notebook descarga el raster de velocidad media de viento (10m) para Costa Rica desde la API oficial de Global Wind Atlas.

**Qué hace:**
- Descarga el archivo GeoTIFF de 250m de resolución
- Verifica que sea un archivo válido
- Lo prepara para subirlo al repositorio ECO-Wind

**Requisitos:**
- Una cuenta de Google (para Colab)
- Conexión a internet normal (GWA está bloqueado en el sandbox de desarrollo)
- ~15 minutos de tiempo

---

## Paso 1: Instalar dependencias

In [ ]:
# Instala rasterio y requests (librerías necesarias para descargar y leer el GeoTIFF)
import subprocess
subprocess.check_call(['pip', 'install', '-q', 'rasterio', 'requests'])
print("✓ Dependencias instaladas")

## Paso 2: Descargar el raster de GWA

In [ ]:
import requests
import os

# URL de la API oficial de Global Wind Atlas (confirmada en el código fuente del paquete R energyRt/globalwindatlas)
url = "https://globalwindatlas.info/api/gis/country/CRI/wind-speed/10"
destino = "gwa_costa_rica_10m.tif"

print(f"Descargando desde:")
print(f"  {url}\n")
print("Esto puede tardar 1-3 minutos (archivo ~80-100 MB)...\n")

r = requests.get(url, stream=True, timeout=120)
r.raise_for_status()

tamaño_total = int(r.headers.get('content-length', 0))
descargado = 0

with open(destino, "wb") as f:
    for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB por vez
        f.write(chunk)
        descargado += len(chunk)
        if tamaño_total:
            pct = 100 * descargado / tamaño_total
            mb_desc = descargado / 1e6
            mb_total = tamaño_total / 1e6
            print(f"Progreso: {pct:5.1f}% ({mb_desc:6.1f} MB / {mb_total:6.1f} MB)", end='\r')

tamaño_mb = os.path.getsize(destino) / 1e6
print(f"\n\n✓ Descargado exitosamente: {destino} ({tamaño_mb:.1f} MB)")

## Paso 3: Verificar el archivo

In [ ]:
import rasterio

# Abre el raster para verificar que sea válido
with rasterio.open(destino) as src:
    print("Información del raster:")
    print(f"  CRS (Sistema de Coordenadas): {src.crs}")
    print(f"  Resolución: {src.width} × {src.height} píxeles")
    print(f"  Bounds (Cobertura):")
    print(f"    Lat: {src.bounds.bottom:.2f}° a {src.bounds.top:.2f}°")
    print(f"    Lon: {src.bounds.left:.2f}° a {src.bounds.right:.2f}°")
    print(f"  Tipo de dato: {src.dtypes[0]}")
    print(f"  Bandas: {src.count}")
    
    # Lee una muestra de píxeles para verificar que hay datos
    muestra = src.read(1)[0:3, 0:3]
    print(f"\n  Muestra de datos (esquina superior izquierda):")
    print(f"    Velocidad media de viento (m/s)")
    for fila in muestra:
        print(f"    {fila}")

print("\n✓ Raster válido y listo para usar")

## Paso 4: Descargar el archivo localmente

El archivo está ahora en Colab. Necesitas descargarlo a tu máquina para subirlo al repositorio.

In [ ]:
from google.colab import files

print("Descargando gwa_costa_rica_10m.tif a tu máquina local...")
print("(Aparecerá una ventana emergente de descarga)\n")

files.download(destino)

print("✓ Descarga completada")

## Paso 5: Subir al repositorio

Ya tienes el archivo descargado. Ahora súbelo al repositorio en GitHub:

### Opción A: Usando Git desde la terminal (recomendado)

```bash
# En tu máquina local:
cd ~/ECO-Wind
cp ~/Descargas/gwa_costa_rica_10m.tif datos_clima/
git add datos_clima/gwa_costa_rica_10m.tif
git commit -m "Add GWA Costa Rica raster (250m resolution from Global Wind Atlas)"
git push origin claude/eco-wind-audit-velocidades-7fv1sy
```

### Opción B: Usando GitHub Web

1. Ve a https://github.com/Sogo2012/ECO-Wind
2. Asegúrate de estar en la rama `claude/eco-wind-audit-velocidades-7fv1sy`
3. Navega a la carpeta `datos_clima/`
4. Haz clic en **Add file → Upload files**
5. Arrastra `gwa_costa_rica_10m.tif`
6. Escribe el mensaje de commit: `Add GWA Costa Rica raster (250m resolution from Global Wind Atlas)`
7. Haz clic en **Commit changes**

---

## ¿Qué pasa después?

Una vez que el archivo esté en el repositorio:

✅ La app automáticamente va a cargar el raster en lugar de fallar  
✅ Tendrás sensibilizaciones con datos reales de GWA (250m de resolución)  
✅ Cobertura total para Costa Rica, no solo los 4 sitios validados  
✅ Ajustes espaciales precisos usando `factor_ajuste_gwa()`

---

**¿Preguntas?** Revisa `engine/gwa_raster.py` en el repositorio (tiene documentación detallada sobre cómo funciona el raster).